In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, when, regexp_replace, trim, concat_ws
from pyspark.ml.feature import StringIndexer
import glob
import os

# Initialize Spark session (limit memory as per proposal)
spark = SparkSession.builder \
    .appName("AmazonMeta2023") \
    .config("spark.driver.memory", "3g") \
    .getOrCreate()


In [ ]:
#!/usr/bin/env python
# coding: utf-8

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, when, regexp_replace, trim, concat_ws
from pyspark.ml.feature import StringIndexer
import glob
import os

# Initialize Spark session (limit memory as our proposal)
spark = SparkSession.builder \
    .appName("AmazonMeta2023") \
    .config("spark.driver.memory", "3g") \
    .getOrCreate()


schema = StructType([
    StructField("main_category", StringType()),
    StructField("title", StringType()),
    StructField("average_rating", DoubleType()),
    StructField("rating_number", LongType()),
    StructField("features", ArrayType(StringType())),
    StructField("description", ArrayType(StringType())),
    StructField("price", DoubleType()),
    StructField("images", ArrayType(StructType([
        StructField("thumb", StringType()),
        StructField("large", StringType()),
        StructField("variant", StringType()),
        StructField("hi_res", StringType())
    ]))),
    StructField("videos", ArrayType(StructType([
        StructField("title", StringType()),
        StructField("url", StringType())
    ]))),
    StructField("store", StringType()),
    StructField("categories", ArrayType(StringType())),
    StructField("details", MapType(StringType(), StringType())),
    StructField("parent_asin", StringType()),
    StructField("bought_together", ArrayType(StringType()))
])

# Load all 22 parts (sorted for consistency)
base_path = r"/home/user/Downloads/3018 project/meta_Electronics.jsonl"
files = sorted(glob.glob(os.path.join(base_path, "part-*.jsonl")))  # Gets part-0001 to part-0022

dfs = []
for file in files:
    df_temp = spark.read.schema(schema).json(file)
    dfs.append(df_temp)

# Union all
df = dfs[0]
for d in dfs[1:]:
    df = df.union(d)

print(f"Total products loaded: {df.count():,}")

# Cleaning logic
df_clean = df \
    .withColumn("brand", col("details")["Brand"]) \
    .withColumn("manufacturer", col("details")["Manufacturer"]) \
    .withColumn("model", col("details")["Item model number"]) \
    .withColumn("dimensions", col("details")["Product Dimensions"]) \
    .withColumn("weight", col("details")["Item Weight"]) \
    .withColumn("color", col("details")["Color"]) \
    .withColumn("material", col("details")["Material"]) \
    .withColumn("country_of_origin", col("details")["Country of Origin"]) \
    .withColumn("date_first_available", col("details")["Date First Available"]) \
    .withColumn("warranty", col("details")["Warranty"]) \
    .withColumn("batteries", col("details")["Batteries"]) \
    .withColumn("batteries_required", col("details")["Batteries Required?"]) \
    .withColumn("included_components", col("details")["Included Components"])

# Clean weight (logic)
df_clean = df_clean \
    .withColumn("weight_lbs",
                when(col("weight").contains("pounds"), regexp_replace(col("weight"), "[^0-9.]", "").cast("double") * 1.0)
                .when(col("weight").contains("ounces"), regexp_replace(col("weight"), "[^0-9.]", "").cast("double") * 0.0625)
                .otherwise(None)) \
    .withColumn("weight_oz", regexp_replace(col("weight"), "[^0-9.]", "").cast("double"))

# Concat texts
df_clean = df_clean.withColumn("features_text", concat_ws(" | ", col("features")))
df_clean = df_clean.withColumn("description_text", concat_ws(" ", col("description")))
df_clean = df_clean.withColumn("categories_text", concat_ws(" > ", col("categories")))

# Select columns (Final selection, simplified)
final_df = df_clean.select(
    "parent_asin", "title", "brand", "manufacturer", "model", "store", "main_category",
    "categories", "categories_text", "average_rating", "rating_number", "price",
    "weight", "weight_lbs", "dimensions", "color", "material", "batteries_required",
    "date_first_available", "features", "features_text", "description_text", "images", "details"
)

# Fill nulls 
fill_rules = {
    "brand": "Unknown", "manufacturer": "Unknown", "main_category": "Others",
    "title": "No Title", "features_text": "", "description_text": "",
    "average_rating": 0.0, "rating_number": 0, "price": -1.0
}
df_filled = final_df.na.fill(fill_rules)

# Drop unused 
cols_to_drop = ["details", "features", "categories", "images"]  # Removed videos/bought_together as they weren't in select
df_optimized = df_filled.drop(*cols_to_drop)

# Index parent_asin to product_id_int (refine with ratings)
indexer = StringIndexer(inputCol="parent_asin", outputCol="product_id_int")
df_ready = indexer.fit(df_optimized).transform(df_optimized)

# Save as single Parquet 
output_path = "meta_electronics.parquet"
df_ready.coalesce(1).write.mode("overwrite").option("compression", "snappy").parquet(output_path)
print(f"Saved metadata to: {os.path.abspath(output_path)}")

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/07 16:07:19 WARN Utils: Your hostname, MyCISC3018-Ubuntu25-DC226952-VirtualBox, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
25/12/07 16:07:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/07 16:07:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Total products loaded: 1,610,012


25/12/07 16:08:06 WARN DAGScheduler: Broadcasting large task binary with size 83.2 MiB
                                                                                

Saved metadata to: /home/user/Downloads/3018 project/meta_electronics.parquet


In [ ]:
# Drop unused (your logic)
cols_to_drop = ["details", "features", "categories", "images"]  # Removed videos/bought_together as they weren't in select
df_optimized = df_filled.drop(*cols_to_drop)

# Index parent_asin to product_id_int (we'll refine this in Step 2 with ratings)
indexer = StringIndexer(inputCol="parent_asin", outputCol="product_id_int")
df_ready = indexer.fit(df_optimized).transform(df_optimized)